# Preamble

## Packages

In [ ]:
import spatialdata as spd
import anndata as ad
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

## Data

In [ ]:
# Directories
project_dir  = Path("/home/janzules/spatial/CAR-T")
data_dir     = project_dir / "data"
results_dir  = project_dir / "results"
figures_dir  = project_dir / "figures"

In [ ]:
# Files
hallmark_file = Path("/home/janzules/spatial/CAR-T/code/references/Mouse_Hallmark.gmt")
zarr_file     = Path("/coh_labs/yunroseli/Jona/CAR-T/data/zarr/CellCharterClusters_c2l_annotated")

### Loading Data

In [ ]:
sdata = spd.read_zarr(zarr_file)
adata = sdata.tables['segmentation_counts']  # verify key if table name differs
adata

## Functions

In [ ]:
def read_gmt(gmt_path, min_genes=3, max_genes=5000):
    pathways = {}
    with open(gmt_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name = parts[0].replace("HALLMARK_", "").lower().capitalize()
            genes = [g for g in parts[2:] if g]  # parts[1] is "description"
            if min_genes <= len(genes) <= max_genes:
                pathways[name] = genes
    return pathways

pathways_dict = read_gmt(hallmark_file)

In [ ]:
# Scoring Pathways of interest

def pathway_scoring(adata, pathways_path, pathways_of_interest):
    # loading pathways
    pathways = read_gmt(pathways_path)
    
    # Chcking if I named the pathways correctly
    failed_pathways = []
    for pathway_name in pathways_of_interest:
        if pathway_name not in pathways:
            print(f"Warning: Pathway '{pathway_name}' not found in the provided pathways.")
            failed_pathways.append(pathway_name)
            continue
    
        #Scoring pathways
        genes = pathways[pathway_name]
        
        genes_in_data = [gene for gene in genes if gene in adata.var_names]
        if len(genes_in_data) < 10: # the lowest number of genes in these pathways is around 27
            print(f"Warning: Not enough genes from pathway '{pathway_name}' are present in the data for scoring.")
            failed_pathways.append(pathway_name)
            continue
        
        sc.tl.score_genes(adata, 
                          gene_list=genes_in_data, 
                          score_name=pathway_name)
        if failed_pathways:
            print(f"Failed to score the following pathways: {', '.join(failed_pathways)}")

def find_pathway_names(pathways, search_term: list):
    matches = [key for key in pathways.keys() if any(term.lower() in key.lower() for term in search_term)]
    return matches

In [ ]:
def plot_pathway_histogram(
    adata,
    pathway,
    groupby=None,
    display=True,
    save=False,
    figures_dir=None
):
    """
    Parameters
    ----------
    pathway     : str — name of the pathway score column in adata.obs.
    groupby     : None  → one histogram for the whole dataset.
                  str   → one histogram per unique value in adata.obs[groupby].
                  e.g. 'treatment' or 'tissue'
    display     : bool — show figure(s) inline in the current cell.
    save        : bool — write figure(s) as PNG to figures_dir/pathwayAnalysis/{pathway}/.
    figures_dir : Path or str — base figures directory; required when save=True.
    """
    if pathway not in adata.obs.columns:
        raise ValueError(f"Pathway '{pathway}' not found in adata.obs. Run pathway_scoring first.")
    
    if save and figures_dir is None:
        raise ValueError("figures_dir must be provided when save=True.")
    
    save_root = Path(figures_dir) / "pathwayAnalysis" / pathway if save else None
    if save and save_root:
        save_root.mkdir(parents=True, exist_ok=True)
    
    def _plot_single(scores, title, fname):
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(scores.dropna(), bins=50, edgecolor="white", linewidth=0.3)
        ax.set_xlabel(f"{pathway} score")
        ax.set_ylabel("Cell count")
        ax.set_title(title)
        plt.tight_layout()
        if save:
            fig.savefig(save_root / fname, dpi=150, bbox_inches="tight")
        if display:
            plt.show()
        else:
            plt.close(fig)
    
    if groupby is None:
        _plot_single(
            adata.obs[pathway],
            title=f"{pathway} \u2014 All cells",
            fname=f"{pathway}_all.png"
        )
    else:
        for condition in sorted(adata.obs[groupby].dropna().unique()):
            mask = adata.obs[groupby] == condition
            _plot_single(
                adata.obs.loc[mask, pathway],
                title=f"{pathway} \u2014 {groupby}: {condition}",
                fname=f"{pathway}_{groupby}_{condition}.png"
            )

# Analysis

## Pathway Scoring

In [ ]:
# Pathways of interest — edit this list to customize
pathways_of_interest = [
    'Glycolysis',
    'Interferon_alpha_response',
    'Interferon_gamma_response',
]

pathway_scoring(adata, hallmark_file, pathways_of_interest)

## Histograms

### Full Dataset

In [ ]:
for pathway in pathways_of_interest:
    plot_pathway_histogram(adata, pathway, groupby=None, display=True, save=False)

### Per Treatment Condition

In [ ]:
for pathway in pathways_of_interest:
    plot_pathway_histogram(adata, pathway, groupby='treatment', display=True, save=False)

### Per Tissue

In [ ]:
for pathway in pathways_of_interest:
    plot_pathway_histogram(adata, pathway, groupby='tissue', display=True, save=False)

### Save All Figures

Run this cell to save all histogram figures as PNGs to `figures/pathwayAnalysis/{pathway}/`.

In [ ]:
for pathway in pathways_of_interest:
    plot_pathway_histogram(adata, pathway, groupby=None,        display=False, save=True, figures_dir=figures_dir)
    plot_pathway_histogram(adata, pathway, groupby='treatment', display=False, save=True, figures_dir=figures_dir)
    plot_pathway_histogram(adata, pathway, groupby='tissue',    display=False, save=True, figures_dir=figures_dir)